In [1]:
# ============================================================
# 04_folium_geographic_map.ipynb
# AIRPORT NETWORK - GEOGRAPHIC MAP (FOLIUM)
# ============================================================

import pickle
from pathlib import Path
import folium
import networkx as nx

# ============================================================
# PATHS
# ============================================================

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "../data"
DOCS_DIR = BASE_DIR / "../docs"

DOCS_DIR.mkdir(exist_ok=True)

# ============================================================
# LOAD GRAPH
# ============================================================

with open(DATA_DIR / "airport_network.gpickle", "rb") as f:
    G = pickle.load(f)

print("Graph loaded:", G.number_of_nodes(), "nodes")

# ============================================================
# CREATE MAP
# ============================================================

m = folium.Map(
    location=[20, 0],
    zoom_start=2,
    tiles="cartodbpositron"
)

# ============================================================
# DRAW EDGES (ROUTES)
# ============================================================

print("Adding routes...")

edge_count = 0

for u, v in G.edges():

    u_data = G.nodes[u]
    v_data = G.nodes[v]

    if (
        "latitude" in u_data and "longitude" in u_data and
        "latitude" in v_data and "longitude" in v_data
    ):

        lat1, lon1 = u_data["latitude"], u_data["longitude"]
        lat2, lon2 = v_data["latitude"], v_data["longitude"]

        folium.PolyLine(
            locations=[[lat1, lon1], [lat2, lon2]],
            weight=0.5,
            opacity=0.05,
            color="blue"
        ).add_to(m)

        edge_count += 1

        if edge_count % 5000 == 0:
            print("Routes added:", edge_count)

print("Total routes:", edge_count)

# ============================================================
# DRAW NODES (AIRPORTS)
# ============================================================

print("Adding airports...")

for node in G.nodes():

    data = G.nodes[node]

    if "latitude" in data and "longitude" in data:

        lat = data["latitude"]
        lon = data["longitude"]

        degree = G.degree(node)

        city = data.get("city", "")
        country = data.get("country", "")

        folium.CircleMarker(
            location=[lat, lon],
            radius=max(2, degree * 0.05),
            color="red",
            fill=True,
            fill_opacity=0.6,
            popup=f"{node}<br>{city}, {country}<br>Degree: {degree}"
        ).add_to(m)

# ============================================================
# SAVE (IMPORTANT: DIFFERENT FILE)
# ============================================================

output_file = DOCS_DIR / "map.html"

m.save(str(output_file))

print("\n✔ Saved geographic map at:")
print(output_file)

print("""
IMPORTANT:
- Your PyVis index.html is untouched
- This is a separate file: map.html
""")

Graph loaded: 3397 nodes
Adding routes...
Routes added: 5000
Routes added: 10000
Routes added: 15000
Total routes: 18908
Adding airports...

✔ Saved geographic map at:
/home/mzhc13/master/complex-networks-airports/notebooks/../docs/map.html

IMPORTANT:
- Your PyVis index.html is untouched
- This is a separate file: map.html

